# LLM From DFM to Star Schema

## Import Libraries

In [ ]:
import sys, os, json, re
from datetime import datetime

sys.path.insert(0, os.path.abspath(""))

from utils.config import *
from utils.llm_client import call_llm, LLM_PROVIDER
from utils.loader import load_tables
import pandas as pd

BASE_PATH = MODELING_REPORT_DIR
LLM_PATH = MODELING_REPORT_DIR / "LLM"
os.makedirs(BASE_PATH, exist_ok=True)
LLM_PATH.mkdir(parents=True, exist_ok=True)

dfs = load_tables("cleaned", normalize_cols="lower")

schema_path = BASE_PATH / "schema.json"
assert schema_path.exists(), f"Missing: {schema_path} - run Classical Modeling first."

with open(schema_path) as f:
    classical_schema = json.load(f)


def is_llm_error(text):
    return isinstance(text, str) and text.startswith("[LLM Error]")


print("Libraries imported from utils. LLM client ready.")
print(f"LLM provider: {LLM_PROVIDER.upper()}")
print(f'  Fact:       {classical_schema["star_schema"]["fact_table"]["name"]}')
print(f'  Measures:   {classical_schema["star_schema"]["fact_table"]["measures"]}')
print(f'  Dimensions: {list(classical_schema["star_schema"]["dimension_tables"].keys())}')
print(f"  LLM output: {LLM_PATH}")

## Fact Table Suggestion

In [ ]:
def build_schema_description(dataframes: dict) -> str:
    """
    Compact textual description of the reconciled DB schema.
    Passed to the LLM as grounding context.
    """
    parts = []
    for name, df in dataframes.items():
        parts.append(f"TABLE {name}:")
        parts.append(f"  rows: {len(df):,}")
        for col in sorted(df.columns):
            dtype = str(df[col].dtype)
            n_unique = df[col].nunique()
            sample = df[col].dropna()
            sample = str(sample.iloc[0])[:30] if not sample.empty else "--"
            parts.append(f"  - {col:25s} {dtype:12s} unique={n_unique:<6d} sample={sample!r}")
        parts.append("")
    return "\n".join(parts)


schema_desc = build_schema_description(
    {
        "Flights (sample 5 rows)": dfs["Flights"].head(5),
        "Aircrafts": dfs["Aircrafts"],
        "Carriers": dfs["Carriers"],
        "Stations (sample)": dfs["Stations"].head(10),
        "Cancellation": dfs["Cancellation"],
        "Active_Weather": dfs["Active_Weather"],
        "Weather_Observations (sample 5 rows)": dfs["Weather_Observations"].head(5),
    }
)

SYSTEM_FACT = """You are a senior data warehouse modeler following the Golfarelli-Rizzi DFM methodology.

Your task is to identify the FACT TABLE in a reconciled database schema for a
US domestic flight operations Data Warehouse (year 2022).

The fact table must satisfy ALL of the following criteria:
1. Represents a clearly defined business event (not a lookup or reference table)
2. Has a fine grain - each row corresponds to one atomic occurrence of that event
3. Contains multiple numeric measures that are meaningful to aggregate
4. References multiple dimension tables via foreign keys (it is the "center" of the star)

Additional modeling hints:
- Lookup tables with few rows (< 10) and only code + description are dimension candidates, not facts
- A table with millions of rows and both FK columns and numeric columns is likely the fact
- Weather readings are context for the event, not the event itself

Respond in this exact format - no extra text:

FACT TABLE: <table_name>
GRAIN: <one precise sentence, e.g. "One row per scheduled flight departure">
REASONING:
- <specific structural reason 1>
- <specific structural reason 2>
- <specific structural reason 3>
MEASURES IDENTIFIED: <comma-separated list of numeric columns that are measures>
DIMENSIONS REFERENCED: <comma-separated list of FK columns or referenced tables>"""

prompt_fact = f"""Given this reconciled database schema for a US domestic airline project,
identify which table should be the fact table in the star schema.

Schema:
{schema_desc}

Apply the Golfarelli-Rizzi DFM criteria strictly.
Pay attention to cardinality (number of rows) and the presence of FK columns."""

response_fact = call_llm(prompt_fact, system=SYSTEM_FACT)
if is_llm_error(response_fact):
    print(f"[Fact Table Suggestion] Skipped - {response_fact}")
    response_fact = ""

print("LLM RESPONSE - Fact Suggestion:")
print()
print(response_fact)
print()
print("CLASSICAL CHOICE:")
print(f'  Fact:  {classical_schema["star_schema"]["fact_table"]["name"]}')
print(f'  Grain: {classical_schema["grain"]}')
print(f'  Match: {"Yes" if "flight" in response_fact.lower() else "REVIEW NEEDED"}')

# Save
if response_fact:
    with open(LLM_PATH / "01_fact_suggestion.txt", "w") as f:
        f.write(response_fact)
    print(f"\nSaved: {LLM_PATH / '01_fact_suggestion.txt'}")
else:
    print("Fact suggestion not saved because the LLM response failed.")

## Measure Additivity Classifier

In [ ]:
measures_to_classify = [
    "dep_delay",
    "taxi_out",
    "air_time",
    "distance",
    "wind_spd",
    "wind_gust",
    "visibility",
    "temperature",
    "cloud_cover",
]

# Build measure metadata - flight cols from Flights, weather cols from Weather_Observations
flight_sample = dfs["Flights"].head(50000)
wo_sample = dfs["Weather_Observations"]

measure_info = []
for m in measures_to_classify:
    src = flight_sample if m in flight_sample.columns else wo_sample
    if m in src.columns:
        col = src[m].dropna()
        entry = {
            "name": m,
            "dtype": str(src[m].dtype),
            "min": round(float(col.min()), 2) if len(col) else None,
            "max": round(float(col.max()), 2) if len(col) else None,
            "mean": round(float(col.mean()), 2) if len(col) else None,
            "sample": col.head(3).tolist(),
        }
        if src is wo_sample:
            entry["source"] = "Weather_Observations (snapshot at origin airport at departure hour)"
        measure_info.append(entry)

SYSTEM_ADDITIVITY = """You are a DW modeler specializing in dimensional modeling (Golfarelli-Rizzi).

Classify the ADDITIVITY of each measure across the standard dimensions of this DW:
time (fl_date), departure hour (dep_hour), origin airport, destination airport,
marketing carrier, operating carrier.

Definitions - apply strictly:
- ADDITIVE:      SUM is meaningful across ALL dimensions without exception
                 (e.g. total delay minutes, total airborne minutes)
- SEMI_ADDITIVE: SUM is meaningful across SOME dimensions but not others
                 (e.g. distance is additive per carrier for network totals,
                  but averaging per route is more meaningful)
- NON_ADDITIVE:  SUM is NEVER meaningful; always use AVG / MIN / MAX
                 (e.g. weather snapshots - temperature, wind speed - are
                  point-in-time readings, not quantities to accumulate)

Return ONLY valid JSON - no explanation outside the JSON block:
{
  "measures": [
    {
      "name": "...",
      "additivity": "additive|semi_additive|non_additive",
      "reasoning": "one concise sentence explaining why",
      "valid_aggs": ["SUM", "AVG", "..."]
    }
  ]
}"""

prompt_add = f"""Classify the additivity of these measures from the FLIGHTS fact table.

Fact table grain: one row per scheduled flight departure (US domestic, 2022).

{json.dumps(measure_info, indent=2)}

Key domain context:
- dep_delay: actual departure minus scheduled departure in minutes (negative = early)
- taxi_out:  gate push-back to wheels-off in minutes
- air_time:  wheels-off to wheels-on in minutes
- distance:  great-circle route distance in statute miles
- wind_spd, wind_gust, visibility, temperature, cloud_cover:
  weather snapshot recorded at the ORIGIN AIRPORT at the DEPARTURE HOUR -
  these are environmental readings, not accumulated quantities"""

response_add = call_llm(prompt_add, system=SYSTEM_ADDITIVITY)
if is_llm_error(response_add):
    print(f"[Measure Additivity] Skipped - {response_add}")
    response_add = ""

print("LLM RESPONSE - Measure Additivity:")
print()
print(response_add)

# Save
if response_add:
    with open(LLM_PATH / "02_additivity.json", "w") as f:
        f.write(response_add)
    print(f"\nSaved: {LLM_PATH / '02_additivity.json'}")
else:
    print("Additivity response not saved because the LLM response failed.")

In [ ]:
CLASSICAL_ADDITIVITY = classical_schema["measures"]["definitions"]

try:
    raw = re.sub(r"```json|```", "", response_add).strip()
    llm_result = json.loads(raw)
    llm_map = {m["name"]: m["additivity"] for m in llm_result.get("measures", [])}

    print("Comparison - Classical vs LLM additivity:")
    print()
    print(f'  {"Measure":<18} {"Classical":<18} {"LLM":<18} Status')
    print("  " + "-" * 65)
    for measure, atype in CLASSICAL_ADDITIVITY.items():
        llm_type = llm_map.get(measure, "not found")
        status = "OK" if atype == llm_type else "REVIEW"
        print(f"  {measure:<18} {atype:<18} {llm_type:<18} {status}")

    # Save comparison as CSV
    rows = [{"measure": m, "classical": CLASSICAL_ADDITIVITY.get(m, "-"), "llm": llm_map.get(m, "not found")} for m in measures_to_classify]
    pd.DataFrame(rows).to_csv(LLM_PATH / "02_additivity_comparison.csv", index=False)
    print(f"\nSaved: {LLM_PATH / '02_additivity_comparison.csv'}")

except Exception as e:
    print(f"[Parse error: {e}]")
    print("Review the raw response above manually.")

## Hierarchy Detector

In [ ]:
dim_attributes = {
    "DIM_FL_DATES": ["fl_date", "day_of_week", "is_holiday", "month", "quarter", "year"],
    # fl_date is the natural key
    # Hierarchy chain: fl_date -> month -> quarter -> year
    # Cross-dimensional (derived from date but not in rollup chain): day_of_week, is_holiday
    "DIM_DEP_HOURS": ["dep_hour", "time_band"],
    # dep_hour: integer 0-23
    # time_band: Morning / Afternoon / Evening / Night (grouping of dep_hour)
    "DIM_STATIONS": ["airport", "city", "state"],
    # airport: 3-letter FAA code (natural key)
    # Role-playing dimension: referenced as both Origin_Station_ID and Dest_Station_ID
    # Hierarchy: airport -> city -> state
    "DIM_CARRIERS": ["code", "description"],
    # code: 2-char IATA carrier code (natural key)
    # Role-playing dimension: referenced as both MKT_Carrier_ID and OP_Carrier_ID
    # description is a descriptive attribute, NOT a hierarchy level
    "DIM_JUNK": ["manufacturer", "width", "cancellation_reason", "weather_description"],
    # Junk dimension: groups low-cardinality descriptors from different sources
    # manufacturer: aircraft manufacturer (Airbus, Boeing, Bombardier, Embraer, Unknown)
    # width: aircraft body type (Narrow-body, Wide-body)
    # cancellation_reason: reason for cancellation (0=not cancelled ... 4=security)
    # weather_description: active weather condition at origin
    # No natural hierarchy - all attributes are independent descriptors
}

SYSTEM_HIERARCHY = """You are a DW modeler applying the Golfarelli-Rizzi DFM methodology.

For each dimension, identify the HIERARCHY:
an ordered sequence of levels from most detailed (finest grain) to most aggregated.

Rules - apply strictly:
1. Hierarchy levels must define meaningful ROLL-UP paths (aggregation paths)
   Example valid roll-up: airport -> city -> state -> country
2. Descriptive attributes are properties of a dimension member, NOT hierarchy levels
   Example: carrier_description, manufacturer, aircraft_age describe a member but do not roll up
3. Boolean flags (is_holiday) and derived attributes (day_of_week) are cross-dimensional
   - they derive from the key but are outside the main roll-up chain
4. If a dimension has no roll-up path (e.g. junk dimensions), state so explicitly
5. Note role-playing dimensions where the same physical table serves multiple roles

For each dimension output:
HIERARCHY <DIM_NAME>:
  Natural key: <column>
  Levels: level1 -> level2 -> ... (or "single level - no roll-up path")
  Balanced: Yes / No
  Cross-dimensional attributes: <list or none>
  Descriptive attributes: <list or none>
  Missing levels suggested: <list or none>
  Role-playing note: <if applicable>"""

prompt_h = f"""Detect hierarchies in these dimension attributes for a US domestic airline DW (2022).

{json.dumps(dim_attributes, indent=2)}

Additional context:
- DIM_STATIONS is a role-playing dimension: the same airport table is referenced
  as both origin and destination in the fact table
- DIM_CARRIERS is a role-playing dimension: the same carrier table is referenced
  as both marketing carrier and operating carrier
- DIM_JUNK groups low-cardinality flags from cancellation and weather lookups"""

response_h = call_llm(prompt_h, system=SYSTEM_HIERARCHY)
if is_llm_error(response_h):
    print(f"[Hierarchy Detector] Skipped - {response_h}")
    response_h = ""

print("LLM RESPONSE - Hierarchy Detector:")
print()
print(response_h)

# Save
if response_h:
    with open(LLM_PATH / "03_hierarchies.txt", "w") as f:
        f.write(response_h)
    print(f"\nSaved: {LLM_PATH / '03_hierarchies.txt'}")
else:
    print("Hierarchy response not saved because the LLM response failed.")

## Glossary Generator

In [ ]:
SYSTEM_GLOSSARY = """You are writing the official Glossary of Measures for a University
Data Warehouse exam project on US domestic airline operations.

Fact table: FLIGHTS - grain: one row per scheduled flight departure (US domestic, 2022).

For each DERIVED measure (e.g. avg_delay = AVG(dep_delay)), produce a compact entry:
- Measure:     the derived measure name as it appears in the glossary
- Definition:  one clear sentence describing what the measure represents
- Unit:        measurement unit (minutes, statute miles, knots, °C, ordinal 0-4, count)
- Computation: exact SQL expression (e.g. AVG(Dep_Delay))
- Additivity:  additive | semi_additive | non_additive
               (this refers to the BASE measure, not the derived aggregation)
- Valid Aggs:  SQL aggregations that are meaningful on the BASE column

Strict constraints - do NOT violate:
- dep_delay, taxi_out, air_time -> base measure is ADDITIVE
- distance -> base measure is SEMI_ADDITIVE (SUM meaningful only as network total)
- wind_spd, wind_gust, visibility, temperature, cloud_cover ->
  base measure is NON_ADDITIVE (weather snapshots - point-in-time environmental readings)
- AVG, MAX, MIN derived measures are themselves NON_ADDITIVE
  (cannot be summed across slices - must be recomputed from base)
- COUNT(*) = # of flights is ADDITIVE

Output as a Markdown table with these exact columns:
| Measure | Definition | Unit | Computation | Additivity | Valid Aggs |"""

glossary_measures = [
    ("# of flights", "COUNT(*)"),
    ("avg_delay", "AVG(Dep_Delay)"),
    ("sum_delay", "SUM(Dep_Delay)"),
    ("max_delay", "MAX(Dep_Delay)"),
    ("avg_taxi_out", "AVG(Taxi_Out)"),
    ("avg_air_time", "AVG(Air_Time)"),
    ("avg_distance", "AVG(Distance)"),
    ("avg_wind_spd", "AVG(Wind_Spd)"),
    ("max_wind_gust", "MAX(Wind_Gust)"),
    ("min_visibility", "MIN(Visibility)"),
    ("avg_temperature", "AVG(Temperature)"),
    ("min_temperature", "MIN(Temperature)"),
    ("avg_cloud_cover", "AVG(Cloud_Cover)"),
]

measures_list = "\n".join(f"- {name}: {comp}" for name, comp in glossary_measures)

prompt_gloss = f"""Produce the Glossary of Measures for FLIGHTS.

Derived measures to document (name: computation):
{measures_list}

Base column context:
- Dep_Delay:    departure delay in minutes (negative = early departure)
- Taxi_Out:     gate push-back to wheels-off in minutes
- Air_Time:     wheels-off to wheels-on in minutes
- Distance:     great-circle route distance in statute miles
- Wind_Spd:     sustained wind speed at origin airport at departure hour (knots)
- Wind_Gust:    peak wind gust at origin at departure hour (knots)
- Visibility:   horizontal visibility at origin at departure hour (statute miles)
- Temperature:  air temperature at origin at departure hour (degrees Celsius)
- Cloud_Cover:  cloud cover index: 0=clear, 1=few, 2=scattered, 3=broken, 4=overcast"""

response_gloss = call_llm(prompt_gloss, system=SYSTEM_GLOSSARY)
if is_llm_error(response_gloss):
    print(f"[Glossary Generator] Skipped - {response_gloss}")
    response_gloss = ""

print("LLM GLOSSARY:")
print()
print(response_gloss)

# Save
if response_gloss:
    with open(LLM_PATH / "04_glossary.md", "w") as f:
        f.write("# Glossary of Measures - FLIGHTS\n\n")
        f.write(f"*Generated by LLM ({LLM_PROVIDER}) - review before use*\n\n")
        f.write(response_gloss)
    print(f"\nSaved: {LLM_PATH / '04_glossary.md'}")
else:
    print("Glossary not saved because the LLM response failed.")

## Schema Validator

In [ ]:
classical_summary = {
    "fact_table": classical_schema["star_schema"]["fact_table"],
    "dimension_tables": classical_schema["star_schema"]["dimension_tables"],
    "measure_additivity": classical_schema["measures"]["definitions"],
    "grain": classical_schema["grain"],
    "special_patterns": [
        "DIM_STATIONS is a role-playing dimension: Origin_Station_ID and Dest_Station_ID both reference DIM_STATIONS(Station_ID)",
        "DIM_CARRIERS is a role-playing dimension: MKT_Carrier_ID and OP_Carrier_ID both reference DIM_CARRIERS(Carrier_ID)",
        "DIM_JUNK groups: Cancellation_Reason, Weather_Description",
        "Aircraft_Age is a descriptive attribute on the fact table, derived at ETL as current_year - year_of_manufacture",
    ],
}

SYSTEM_VALIDATE = """You are a peer reviewer of a Data Warehouse star schema built with
the Golfarelli-Rizzi DFM methodology. Your role is to provide constructive,
specific feedback - not generic advice.

Evaluate the schema on these points:
1. Fact choice and grain definition - is the grain precise and unambiguous?
2. Measure additivity - is each measure correctly classified?
   Pay special attention to weather snapshot measures (non-additive) and distance (semi-additive).
3. Hierarchy completeness - are roll-up paths complete for each dimension?
4. Role-playing dimensions - are DIM_STATIONS and DIM_CARRIERS correctly modeled?
5. Junk dimension - are the grouped attributes coherent (low cardinality, no natural hierarchy)?
6. Aircraft_Age on fact - is this the right placement, or should it be in a dimension?
7. Missing dimensions or attributes - anything important omitted?

For each finding use exactly one severity label:
- INFO:     minor documentation or naming suggestion
- WARNING:  design choice worth discussing with justification
- CRITICAL: clear modeling error that violates DFM principles

End with a one-line overall verdict: APPROVED / APPROVED WITH RESERVATIONS / NEEDS REVISION"""

prompt_v = f"""Peer-review this star schema for a US domestic airline flight DW (2022).

{json.dumps(classical_summary, indent=2, default=str)}

Provide specific findings with INFO / WARNING / CRITICAL labels.
End with your overall verdict."""

response_v = call_llm(prompt_v, system=SYSTEM_VALIDATE)
if is_llm_error(response_v):
    print(f"[Schema Validator] Skipped - {response_v}")
    response_v = ""

print("LLM PEER REVIEW:")
print()
print(response_v)

# Save
if response_v:
    with open(LLM_PATH / "05_peer_review.txt", "w") as f:
        f.write(response_v)
    print(f"\nSaved: {LLM_PATH / '05_peer_review.txt'}")
else:
    print("Peer review not saved because the LLM response failed.")

## Save Enriched Schema

In [ ]:
enriched = {
    "generated_at": datetime.now().isoformat(),
    "source": "8-LLM_From_DFM_to_Star_Schema.ipynb",
    "llm_provider": LLM_PROVIDER,
    "classical_schema": classical_schema,
    "llm_suggestions": {
        "provider": LLM_PROVIDER,
        "01_fact_suggestion": response_fact,
        "02_additivity": response_add,
        "03_hierarchies": response_h,
        "04_glossary": response_gloss,
        "05_peer_review": response_v,
    },
    "decision_log": [
        "Review each LLM suggestion before committing to the schema.",
        "If LLM disagrees with classical schema, document the decision and rationale.",
        "Final schema.json for the ETL phase must be the one the human modeler approves.",
        "LLM responses are advisory only - they do not replace domain validation.",
        "Weather measures must be non_additive - flag any LLM disagreement on this.",
        "Role-playing dimensions (DIM_STATIONS, DIM_CARRIERS) must be confirmed correct by reviewer.",
    ],
}

enriched_path = LLM_PATH / "schema_with_llm.json"
with open(enriched_path, "w") as f:
    json.dump(enriched, f, indent=2, default=str)

print(f"schema_with_llm.json saved: {enriched_path}")
print(f"  Size: {os.path.getsize(enriched_path):,} bytes")
print()
print("LLM outputs saved in:", LLM_PATH)
print("  01_fact_suggestion.txt")
print("  02_additivity.json")
print("  02_additivity_comparison.csv")
print("  03_hierarchies.txt")
print("  04_glossary.md")
print("  05_peer_review.txt")
print("  schema_with_llm.json")